# Error Handling
**Topic:** Python Fundamentals

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Identify** the most common Python exception types and what triggers each one
- **Explain** how `try`/`except`/`finally` lets a program recover from an error instead of crashing
- **Interpret** when to catch a specific exception type versus a broad `Exception` and why the distinction matters

---
## How we got here

In *08: Object-Oriented Programming* we saw that calling `predict()` before `fit()` raises a `RuntimeError`. That was our first deliberately raised exception. In real pipelines, exceptions happen constantly: files do not exist, columns have the wrong type, API responses are malformed. This notebook gives you the tools to handle those situations gracefully instead of letting them crash your entire pipeline.

---
## Why this matters for data science

A data pipeline that crashes on the first bad row is not production-ready. Real pipelines wrap file reads in `try/except FileNotFoundError`, validate column types before transformations, and log problematic rows rather than silently dropping them or halting entirely. Understanding error handling is also what lets you write useful error messages: raising `ValueError("Expected a numeric column, got dtype=object")` is far more helpful to a teammate than watching the pipeline crash three steps later with a confusing `AttributeError`.

---
## Try it yourself

In [2]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        return 'Error: cannot divide by zero'
    except TypeError as e:
        return f'Error: wrong type — {e}'
    else:
        return result
    finally:
        print(f'safe_divide({a}, {b}) was called')

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, 'x'))

safe_divide(10, 2) was called
5.0
safe_divide(10, 0) was called
Error: cannot divide by zero
safe_divide(10, x) was called
Error: wrong type — unsupported operand type(s) for /: 'int' and 'str'


In [3]:
# ✏️ Your turn — modify this code:
# 1. Change 'unknown' to None and see which except branch catches it
# 2. Add a ValueError branch for when the value can't be converted to float
# 3. What happens if you add 'raise' at the end of an except block?

def parse_value(raw):
    try:
        return float(raw)
    except TypeError:
        print(f'TypeError: received {type(raw).__name__}')
        return None
    except ValueError:
        return 'Error: value cannot be converted to float.'
    

print(parse_value('3.14'))
print(parse_value(None))
print(parse_value(None))

3.14
TypeError: received NoneType
None
TypeError: received NoneType
None


In [ ]:
# 🎯 Challenge:
# Wrap this code in a try/except block that handles both KeyError and TypeError
# with different, informative error messages for each:
#
#   data = {'name': 'Alice', 'age': 30}
#   result = data['salary'] + 1000
#
# Hint: use two separate except clauses, one for each exception type

# Your code here:

---
## What's happening?

When Python encounters an operation it cannot complete, it raises an **exception**, an object that describes what went wrong. If nothing catches it, the program halts and prints a traceback. The `try`/`except` block lets you intercept the exception and decide what to do instead.

| Exception type | Common trigger in data science | How to handle it |
|----------------|-------------------------------|-----------------|
| `FileNotFoundError` | CSV path does not exist | Check path, log warning, skip file |
| `KeyError` | Column name not in DataFrame | Check `df.columns` first; use `.get()` |
| `ValueError` | Wrong value for an operation (e.g., log of negative) | Validate input before the call |
| `TypeError` | Wrong type passed to a function | Check dtype before the operation |
| `AttributeError` | Called a method that does not exist on this object | Check that the object is the right type |

```python
def load_csv_safely(path):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Warning: {path} not found. Returning empty DataFrame.")
        return pd.DataFrame()
    except pd.errors.ParserError as e:
        print(f"Could not parse {path}: {e}")
        return pd.DataFrame()
    else:
        print(f"Loaded {len(df)} rows from {path}.")
        return df
    finally:
        print("load_csv_safely() finished.")  # always runs
```

### Catch specific exceptions, not bare `except`

`except Exception:` catches everything, including `KeyboardInterrupt` and `SystemExit`, which are signals you usually want to propagate. `except ValueError:` catches exactly what you expect and lets everything else bubble up, which makes bugs easier to find.

Return to the widget and switch to the "handled version" for each error type to compare how the specific handler changes the behavior.

---
## A direct example: parsing a list that contains bad values

Six price strings, two of them unparseable. Without error handling the loop crashes on `"N/A"`. With `try/except`, the function returns `None` for bad values and keeps going.

- **Notice:** All six items are processed even though two raise a `ValueError` — the pipeline does not stop
- **Notice:** Returning `None` for failures makes the bad rows visible and filterable instead of silently dropped
- **Notice:** This is the same pattern used in any data loading function that should not crash on a single bad row

In [ ]:
items      = ["A", "B", "C", "D", "E", "F"]
raw_prices = ["$12.99", "$8.50", "N/A", "$24.00", "free", "$6.75"]

def parse_price(value):
    try:
        return float(value.replace("$", ""))
    except ValueError:
        return None

prices = [parse_price(v) for v in raw_prices]
print(list(zip(raw_prices, prices)))

colors  = ["#E45756" if p is None else "#4C72B0" for p in prices]
heights = [p if p is not None else 0 for p in prices]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(items, heights, color=colors)
ax.set_title("parse_price(): blue = parsed successfully, red = ValueError caught", fontsize=12)
ax.set_xlabel("Item")
ax.set_ylabel("Price ($)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Real-world example: A data pipeline that handles bad rows gracefully

A batch pipeline processes daily transaction files. Some files are missing, some rows have the wrong number of fields, and some amounts are non-numeric. The chart shows the outcome of running 20 files through a fault-tolerant loading function.

Notice:

- **Notice:** Without error handling, the first bad file would stop the entire run; with handling, the pipeline loads what it can and logs what it skipped
- **Notice:** The majority of failures are `FileNotFoundError` because files from weekends and holidays are legitimately absent, not because of a bug
- **Notice:** `ValueError` failures are the ones worth investigating: they indicate that a file exists but contains data that does not match the schema

> **Discussion question:** The pipeline currently prints a warning for each failure. In a production system, what would you do instead of printing, and how would you alert a data engineer if the number of failures on a given day exceeds a threshold?

In [ ]:
np.random.seed(31)

# ── Simulated pipeline run: 20 files, some bad ────────────────────────────────
outcomes = {
    "Loaded successfully":   13,
    "FileNotFoundError":      4,
    "ValueError (bad data)":  2,
    "ParserError":            1,
}
colors = ["#55A868", "#8172B2", "#E45756", "#8B0000"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(list(outcomes.keys()), list(outcomes.values()), color=colors)
for bar, count in zip(bars, outcomes.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(count), ha="center", va="bottom", fontsize=11)
ax.set_title("Daily Transaction Pipeline: Outcomes Across 20 Files", fontsize=13)
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of Files")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### Exception handling patterns for data pipelines

| Pattern | Code sketch | When to use it |
|---------|------------|---------------|
| Skip and log | `except ValueError: logger.warning(...)` | Non-critical rows; pipeline must finish |
| Return a default | `except KeyError: return default_value` | Optional fields; safe fallback exists |
| Re-raise with context | `except Exception as e: raise RuntimeError("Pipeline failed") from e` | Critical step; caller needs to know |
| Validate before the call | `if col not in df.columns: raise KeyError(col)` | Fail early with a clear message |
| Finally for cleanup | `finally: conn.close()` | Database connections, file handles |

---
## Key takeaway

> **Error handling separates a script that works on clean data from a pipeline that works in production; catching specific exceptions, logging what failed, and continuing where possible is what makes data systems reliable.**

---
*Next up: Dates & Times — parsing, formatting, and resampling the timestamps that appear in almost every real-world dataset*